# Day 11 · Exercise 4: Build a Document Index

**What you'll build:** `build_index(docs: list[str], model: str) -> list[dict]` — a function that embeds every document once at index time and returns a list of `{"text": str, "embedding": list[float]}` dicts ready for repeated search.

**Why it matters:** Pre-computing embeddings at index time is the architectural move that makes semantic search practical — a corpus of 1,000 chunks needs one embedding call per query, not 1,000.

## Your Implementation

In [ ]:
import ollama


def build_index(docs: list[str], model: str) -> list[dict]:
    """Embed each document once and return a list of index entries.

    Args:
        docs:  The document strings to index (one entry per string).
        model: The Ollama embedding model to use, e.g. 'nomic-embed-text'.

    Returns:
        A list of dicts, one per document, each with exactly two keys:
          - 'text'      (str)         — the original document string
          - 'embedding' (list[float]) — the vector produced by the model

    Example:
        index = build_index(
            ["Python is a language.", "Dogs are friendly."],
            model="nomic-embed-text",
        )
        # index[0] == {"text": "Python is a language.", "embedding": [...]}
        # index[1] == {"text": "Dogs are friendly.",    "embedding": [...]}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

DOCS = [
    "Python is a high-level programming language.",
    "Dogs are loyal and friendly animals.",
    "Machine learning models learn from data.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by the human brain.",
]
MODEL = "nomic-embed-text"


def _run_checks():
    score, total = 0, 4

    # Check 1: build_index is callable and returns a list
    try:
        assert callable(build_index), "build_index is not defined"
        result = build_index(DOCS, MODEL)
        assert isinstance(result, list), (
            f"build_index must return a list, got {type(result).__name__}"
        )
        print(f"{_PASS} Check 1/{total}: build_index is callable and returns a list")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1/{total}: {e}")
        return  # later checks all depend on a valid index

    # Check 2: index length matches number of documents
    try:
        assert len(result) == len(DOCS), (
            f"expected {len(DOCS)} entries, got {len(result)}"
        )
        print(f"{_PASS} Check 2/{total}: index has one entry per document ({len(DOCS)} entries)")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 2/{total}: {e}")

    # Check 3: every entry has 'text' (str) and 'embedding' (list[float]) keys
    try:
        for i, entry in enumerate(result):
            assert isinstance(entry, dict), f"entry {i} is not a dict"
            assert "text" in entry, f"entry {i} missing 'text' key"
            assert "embedding" in entry, f"entry {i} missing 'embedding' key"
            assert isinstance(entry["text"], str), (
                f"entry {i}['text'] must be str, got {type(entry['text']).__name__}"
            )
            assert isinstance(entry["embedding"], list), (
                f"entry {i}['embedding'] must be list, got {type(entry['embedding']).__name__}"
            )
            assert len(entry["embedding"]) > 0, f"entry {i}['embedding'] is empty"
            assert isinstance(entry["embedding"][0], float), (
                f"entry {i}['embedding'] elements must be float"
            )
        print(
            f"{_PASS} Check 3/{total}: every entry has 'text' (str) and 'embedding' (list[float])"
        )
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 3/{total}: {e}")

    # Check 4: documents are NOT re-embedded during a second build_index call
    #   Patch ollama.embeddings to count calls; the count must equal len(DOCS)
    #   (i.e. exactly one call per document, not more).
    try:
        import ollama as _ollama

        _real_embeddings = _ollama.embeddings
        _call_count = 0

        def _counting_embeddings(model, prompt):
            nonlocal _call_count
            _call_count += 1
            return _real_embeddings(model=model, prompt=prompt)

        _ollama.embeddings = _counting_embeddings
        try:
            build_index(DOCS, MODEL)
        finally:
            _ollama.embeddings = _real_embeddings

        assert _call_count == len(DOCS), (
            f"expected exactly {len(DOCS)} embed calls (one per doc), got {_call_count}. "
            "Are you accidentally re-embedding docs inside the loop more than once?"
        )
        print(
            f"{_PASS} Check 4/{total}: build_index embeds each document exactly once "
            f"({_call_count}/{len(DOCS)} calls)"
        )
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 4/{total}: {e}")

    print()
    if score == total:
        print("=" * 52)
        print(f"  {_PASS}  Exercise complete! {total}/{total} checks passed.")
        print("=" * 52)
    else:
        print(f"  {score}/{total} passed. Keep going!")


_run_checks()

## Bonus Challenge

Extend the index without rebuilding it. Write a function:

```python
def add_to_index(index: list[dict], new_docs: list[str], model: str) -> list[dict]:
    ...
```

that embeds only the new documents and appends them to the existing index, leaving the already-embedded entries untouched. This foreshadows how production vector stores handle incremental updates — only new or changed chunks are re-embedded, keeping costs proportional to the delta, not the full corpus.

(Note: a production async version of this function would require an async-capable embedding client — a pattern introduced in a later day.)

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama


def build_index(docs: list[str], model: str) -> list[dict]:
    """Embed each document once and return a list of index entries."""
    index = []
    for doc in docs:
        response = ollama.embeddings(model=model, prompt=doc)
        index.append({"text": doc, "embedding": response["embedding"]})
    return index
```

**Why this works:** Each document string is passed to `ollama.embeddings` exactly once at build time, and the returned vector is stored alongside the original text in a plain dict. The list of dicts is the entire index — there is no special data structure. At query time a search function embeds only the query and compares it against the already-stored vectors, so the per-document embedding cost is paid once no matter how many times the index is searched.
</details>